In [9]:

import pandas as pd

from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import optuna




In [10]:
df_org_XGB=pd.read_csv('Data/Bank Customer Churn Prediction.csv')
df_org_XGB=df_org_XGB.drop('customer_id',axis=1)

In [11]:
y=df_org_XGB['churn']
X=df_org_XGB.drop('churn',axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
def feature_in_XGB(df):
    col_to_drop=['credit_card','estimated_salary']
    df=df.drop(col_to_drop,axis=1)
    return df
X_train=feature_in_XGB(X_train)
X_test=feature_in_XGB(X_test)
num_X_lr = X_train.select_dtypes(include=['int64','float64']).columns
cat_X_lr = X_train.select_dtypes(include=['object','string']).columns

num_piline=Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
cat_piline=Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
prepecesing_XGB=ColumnTransformer([
    ('num_piline', num_piline, num_X_lr),
    ('cat_piline', cat_piline, cat_X_lr),
])
model_XGB=Pipeline([('preporc',prepecesing_XGB),
                   ('m_XGB',XGBClassifier(n_estimators = 1067, max_depth = 6, learning_rate = 0.082, subsample= 0.846, colsample_bytree=0.6275, gamma= 4.58, min_child_weight = 6, reg_alpha = 4.832, reg_lambda =  0.1434, scale_pos_weight= 2.920))
                   ])
model_XGB


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preporc', ...), ('m_XGB', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_piline', ...), ('cat_piline', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [12]:
for test in ["precision", "recall", "roc_auc"]:
    punkty_model_XGB=cross_val_score(estimator=model_XGB,X=X_train,y=y_train, cv=5,scoring=test)
    print(test,": ", punkty_model_XGB.mean().round(4))

precision :  0.5628
recall :  0.7018
roc_auc :  0.8675


## startowe:
precision :  0.6934
recall :  0.4871
roc_auc :  0.8451

po fe precision :  0.6903
recall :  0.4994
roc_auc :  0.8454

po hiperparametrach
precision :  0.5628
recall :  0.7018
roc_auc :  0.8675

In [13]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.5, 5)
    }

    model = Pipeline([
        ('preporc', prepecesing_XGB),
        ('m_XGB', XGBClassifier(
            **params,
            eval_metric='auc',
            random_state=42,
            n_jobs=-1
        ))
    ])

    score = cross_val_score(model, X_train, y_train,
                            scoring='roc_auc',
                            cv=5).mean()

    return score


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(study.best_params)

[I 2026-05-08 13:48:38,337] A new study created in memory with name: no-name-871d7640-9fd9-44b5-96b9-d483543d3a8d
[I 2026-05-08 13:48:39,706] Trial 0 finished with value: 0.8660457859406151 and parameters: {'n_estimators': 521, 'max_depth': 10, 'learning_rate': 0.0571675561771342, 'subsample': 0.8627581921797698, 'colsample_bytree': 0.7158283354486469, 'gamma': 3.30019977465891, 'min_child_weight': 5, 'reg_alpha': 6.974284041192078, 'reg_lambda': 9.550736172301125, 'scale_pos_weight': 2.188420230885385}. Best is trial 0 with value: 0.8660457859406151.
[I 2026-05-08 13:48:42,669] Trial 1 finished with value: 0.8535422946904104 and parameters: {'n_estimators': 1058, 'max_depth': 10, 'learning_rate': 0.26342780476697764, 'subsample': 0.7348269256884367, 'colsample_bytree': 0.6738172637414038, 'gamma': 1.8217767430717746, 'min_child_weight': 5, 'reg_alpha': 8.179847754638429, 'reg_lambda': 0.7310130736368503, 'scale_pos_weight': 4.538864451162654}. Best is trial 0 with value: 0.86604578594

{'n_estimators': 559, 'max_depth': 6, 'learning_rate': 0.04535749694139665, 'subsample': 0.6075634230037282, 'colsample_bytree': 0.6903168357437789, 'gamma': 4.519650943259898, 'min_child_weight': 7, 'reg_alpha': 0.4904483780771246, 'reg_lambda': 0.13251000707171734, 'scale_pos_weight': 1.8238612904838467}


In [14]:
print(study.best_params)

{'n_estimators': 559, 'max_depth': 6, 'learning_rate': 0.04535749694139665, 'subsample': 0.6075634230037282, 'colsample_bytree': 0.6903168357437789, 'gamma': 4.519650943259898, 'min_child_weight': 7, 'reg_alpha': 0.4904483780771246, 'reg_lambda': 0.13251000707171734, 'scale_pos_weight': 1.8238612904838467}
